In [ ]:
import httpx
import chromadb
import voyageai
import anthropic

* Initialize

In [ ]:
voyage_client = voyageai.Client()  # needs VOYAGE_API_KEY env var  # Embedding
claude_client = anthropic.Anthropic()  # needs ANTHROPIC_API_KEY env var
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection("trials")

* Fetch

In [ ]:
async def fetch_trials(condition: str, page_size: int = 10):
    url = "https://clinicaltrials.gov/api/v2/studies"
    params = {"query.cond": condition, "pageSize": page_size}

    async with httpx.AsyncClient() as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        return response.json()

* Chunk - each study into one text blob (1-doc-per-trial chunking)

In [ ]:
def study_to_text(study):
    protocol = study["protocolSection"]
    identification = protocol["identificationModule"]
    status = protocol.get("statusModule", {})
    description = protocol.get("descriptionModule", {})

    return (
        f"Trial: {identification.get('briefTitle')}\n"
        f"Status: {status.get('overallStatus')}\n"
        f"Summary: {description.get('briefSummary', '')}"
    ), identification["nctId"]

* Embed And Store

In [ ]:
async def index_trials(condition: str):
    data = await fetch_trials(condition)

    texts, ids = [], []
    for study in data["studies"]:
        text, nct_id = study_to_text(study)
        texts.append(text)
        ids.append(nct_id)

    embeddings = voyage_client.embed(texts, model="voyage-3.5", input_type="document").embeddings

    collection.upsert(ids=ids, embeddings=embeddings, documents=texts)
    print(f"Indexed {len(texts)} trials")

* Retrieve

In [ ]:
def retrieve_context(query: str, k: int = 3):
    query_embedding = voyage_client.embed([query], model="voyage-3.5", input_type="query").embeddings[0]

    results = collection.query(query_embeddings=[query_embedding], n_results=k)
    return results["documents"][0]  # list of matched text chunks

* Generate

In [ ]:
def answer_with_rag(query: str):
    context_chunks = retrieve_context(query)
    context = "\n\n---\n\n".join(context_chunks)

    response = claude_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer using only the context above."
        }]
    )
    return response.content[0].text

* Sample output

In [ ]:
# await index_trials("diabetes")
# print(answer_with_rag("Which trials are actively recruiting?"))